# 美国10年期国债收益率时间序列分析

## 项目概述
本notebook对美国10年期国债收益率（DGS10）进行数据获取、探索性分析和时间序列建模预测。

## 数据来源
- **数据源**: FRED (Federal Reserve Economic Data) - 圣路易斯联邦储备银行
- **指标**: DGS10 (10-Year Treasury Constant Maturity Rate)
- **时间范围**: 2000-01-01 至今
- **说明**: 国债收益率是衡量美国政府借款成本的核心指标，也是全球金融市场的重要基准利率

In [ ]:
import requests
import os
from dotenv import load_dotenv

# 从 .env 文件加载环境变量（FRED_API_KEY）
load_dotenv()

# 从环境变量获取API密钥（免费注册获取：https://fred.stlouisfed.org/docs/api/api_key.html）
API_KEY = os.getenv("FRED_API_KEY")

# FRED API 端点 - 获取指定序列的观测值
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

# 请求参数
params = {
    "series_id": "DGS10",        # 美国10年期国债收益率（联邦储备银行官方代码）
    "api_key": API_KEY,          # 你的FRED API密钥
    "file_type": "json",         # 返回JSON格式数据（相比CSV更易于Python处理）
    "observation_start": "2000-01-01",  # 数据起始日期
}

# 发送GET请求获取数据
response = requests.get(BASE_URL, params=params)

# 检查请求是否成功（HTTP 200），失败则抛出异常
response.raise_for_status()

# 解析JSON响应，数据结构包含 'realtime_start', 'realtime_end', 'observation_start', 'observations' 等字段
data = response.json()

print(data)

## 1. 数据获取

## 2. 数据预处理
将API返回的JSON数据转换为pandas DataFrame，设置日期索引，并处理缺失值

## 3. 探索性数据分析 (EDA)
通过可视化展示收益率的时间变化趋势及分布特征

## 4. 时间序列建模 (ARIMA)

### 方法说明
- **ADF检验**: 检验时间序列是否平稳（p-value < 0.05 则拒绝非平稳假设）
- **ARIMA模型**: 自回归积分滑动平均模型，参数(p,d,q)
  - p: 自回归项数
  - d: 差分阶数  
  - q: 滑动平均项数
- **评估指标**: MAE (平均绝对误差)

## 5. 预测结果对比
将模型预测值与实际测试数据进行可视化比较

In [ ]:
# 数据预处理
import pandas as pd

# 从API响应中提取observations列表（每个元素包含date, value, realtime_start）
observations = data['observations']

# 转换为DataFrame，默认列名为date, value, realtime_start, realtime_end
df = pd.DataFrame(observations)

# 将date列转换为datetime类型，便于时间序列操作
df['date'] = pd.to_datetime(df['date'])

# 将value列转换为数值类型，errors='coerce'将无效值（如'.'）转为NaN
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 设置date为索引，便于时间序列切片和绘图
df = df.set_index('date')

# 移除多余的realtime_start列（FRED数据版本的标记，建模时不需要）
df = df.drop('realtime_start', axis=1)

# 查看预处理后的数据前5行
df.head()

In [ ]:
# 数据可视化
import matplotlib.pyplot as plt

# 创建画布，设置尺寸为14x6英寸
plt.figure(figsize=(14, 6))

# 绘制收益率时间序列曲线
# linewidth=0.8 控制线条粗细
plt.plot(df.index, df['value'], linewidth=0.8)

# 设置图表标题和坐标轴标签
plt.title('美国10年期国债收益率 (2000-至今)')
plt.xlabel('日期')
plt.ylabel('收益率 (%)')

# 添加网格线，alpha=0.3设置透明度
plt.grid(True, alpha=0.3)

# 显示图表
plt.show()

# 输出收益率的基本统计量（均值、标准差、最小值、最大值、分位数等）
print(df['value'].describe())

In [ ]:
# 时间序列模型 - ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# ADF平稳性检验
adf_result = adfuller(df['value'].dropna())
print(f'ADF统计量: {adf_result[0]:.4f}')
print(f'p-value: {adf_result[1]:.4f}')
print('数据平稳' if adf_result[1] < 0.05 else '数据非平稳')

# 训练/测试集划分
train = df['value'][:'2020-01-01']
test = df['value']['2020-01-01':]

# ARIMA模型
model = ARIMA(train, order=(2, 1, 2))
model_fit = model.fit()

# 预测
predictions = model_fit.forecast(steps=len(test))
mae = (test - predictions).abs().mean()
print(f'\n测试集MAE: {mae:.4f}')

In [ ]:
# 预测结果可视化
plt.figure(figsize=(14, 6))
plt.plot(test.index, test.values, label='实际值', linewidth=1)
plt.plot(test.index, predictions.values, label='预测值', linewidth=1, alpha=0.7)
plt.title('ARIMA(2,1,2) 预测 vs 实际')
plt.xlabel('日期')
plt.ylabel('收益率 (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. 结论与分析

### 分析总结
1. **数据特征**: 美国10年期国债收益率在2000年后经历了从高点约6.5%到2020年低点接近0.5%的长期下行趋势
2. **平稳性**: ADF检验结果决定是否需要差分处理
3. **预测效果**: ARIMA模型的MAE指标衡量预测精度

### 局限性
- ARIMA模型是统计模型，不考虑宏观经济变量、政策因素等外生变量
- 极端事件（如2008金融危机、2020新冠疫情）会导致收益率剧烈波动，难以准确预测
- 长期预测精度会随时间推移而下降

### 改进方向
- 可尝试GARCH模型捕捉波动率聚集效应
- 可加入宏观经济变量（CPI、GDP增速等）构建VAR或Prophet模型
- 可进行参数优化（AIC/BIC准则）选择最优ARIMA阶数